# Kinetiq batch sources: SYNTHETIC data generator (Databricks / Unity Catalog)

Generates the two **batch** sources that live in Databricks for UC1, *Athletic Marketplace Inventory Rebalancing*:

| Table | Requirement | Grain |
|---|---|---|
| `sell_through_daily` | DS-01 | business_date x location x style-color-size x batch |
| `digital_intent_daily` | DS-02 | business_date x style-color x market x batch |

All rows are **SYNTHETIC** (`record_label = 'SYNTHETIC'`). Kinetiq is a fictional brand.

**Self-contained:** all generator code is in this notebook (no package, no `%pip install`; only the numpy and pandas
that ship with the Databricks runtime). Generation is deterministic: the same parameters give the same rows
(each market x month chunk has its own random stream keyed by seed, market and month).

**Modes:**
- `write`: (re)create the Unity Catalog tables and load them.
- `verify`: regenerate in memory and compare row counts and a content checksum with the existing tables;
  nothing is written. Use it to prove that the tables still match the parameters.

**Row counts:** the natural size comes from the dimensions (markets x stores x styles x history days).
`max_rows_<table>` (0 = no cap) keeps a deterministic random sample if you need a smaller table; the cap is shared
fairly across markets. Capping breaks consistency with the ERP/WMS stock levels, so leave it at 0 for the demo.

**Prerequisite:** a catalog whose managed location is on the private ADLS account (`setup_unity_catalog.py`).

In [ ]:
dbutils.widgets.dropdown("mode", "write", ["write", "verify"], "0. mode")
dbutils.widgets.dropdown("profile", "demo", ["tiny", "small", "demo", "large"], "1. size profile")
dbutils.widgets.text("seed", "", "2. seed (blank = 20261106)")
dbutils.widgets.text("as_of", "", "3. as-of date YYYY-MM-DD (blank = 2026-10-31)")
dbutils.widgets.text("history_days", "", "4. history days 75-1095 (override)")
dbutils.widgets.text("n_markets", "", "5. markets 5-10 (override)")
dbutils.widgets.text("stores_per_market", "", "6. stores per market 3-60 (override)")
dbutils.widgets.text("n_styles", "", "7. styles 12-200 (override)")
dbutils.widgets.text("demand_scale", "", "8. demand scale (override)")
dbutils.widgets.text("dq_defect_rate", "", "9. defect rate 0-0.05 (override)")
dbutils.widgets.text("max_rows_sell_through_daily", "0", "a. max rows sell_through_daily (0 = all)")
dbutils.widgets.text("max_rows_digital_intent_daily", "0", "b. max rows digital_intent_daily (0 = all)")
dbutils.widgets.text("catalog", "appareliq", "c. catalog")
dbutils.widgets.text("schema", "kinetiq_batch", "d. schema")

## Shared world model (identical in notebooks 01 and 02)

The next four cells define the fictional Kinetiq world: static reference lists, parameters, master data
(markets, stores, DCs, styles, SKUs, assortment, prices, partners, lanes, people, events, scenario) and the
expected-demand model. Notebook `02_load_erp_wms_sql` contains **the same code**, so with the same parameters both
notebooks agree on every key and the ERP/WMS stock is consistent with the Databricks sell-through history.
Keep the two copies in sync when you change anything here.

In [ ]:
# Reference data: static lists. Cities are real places; every brand, product, store, partner, vendor
# and person name is invented.
import hashlib
import json
import math
import time
import zlib
from dataclasses import asdict, dataclass, field, fields
from datetime import date, datetime, timedelta, timezone
from datetime import time as dtime

import numpy as np
import pandas as pd


# market_id, name, region, currency, usd->local fx, price premium vs US, hemisphere, timezone, price rounding
MARKETS = [
    ("US", "United States", "NORTH_AMERICA", "USD", 1.00, 1.00, "N", "America/New_York", "x.99"),
    ("GB", "United Kingdom", "EUROPE", "GBP", 0.79, 0.98, "N", "Europe/London", "x.99"),
    ("JP", "Japan", "ASIA", "JPY", 150.0, 1.06, "N", "Asia/Tokyo", "100"),
    ("BR", "Brazil", "LATAM", "BRL", 5.05, 1.45, "S", "America/Sao_Paulo", "x.99"),
    ("AU", "Australia", "APAC", "AUD", 1.52, 1.10, "S", "Australia/Sydney", "x.99"),
    ("DE", "Germany", "EUROPE", "EUR", 0.92, 1.05, "N", "Europe/Berlin", "x.99"),
    ("FR", "France", "EUROPE", "EUR", 0.92, 1.05, "N", "Europe/Paris", "x.99"),
    ("CA", "Canada", "NORTH_AMERICA", "CAD", 1.36, 1.02, "N", "America/Toronto", "x.99"),
    ("CN", "China", "GREATER_CHINA", "CNY", 7.20, 1.00, "N", "Asia/Shanghai", "x9"),
    ("KR", "South Korea", "ASIA", "KRW", 1350.0, 1.02, "N", "Asia/Seoul", "1000"),
]

# market -> [(city, state/province, lat, lon, timezone, [streets/districts])]; first city hosts the flagship
CITIES = {
    "US": [("New York", "NY", 40.7580, -73.9855, "America/New_York", ["Fifth Avenue", "SoHo", "Brooklyn Williamsburg"]),
           ("Los Angeles", "CA", 34.0522, -118.2437, "America/Los_Angeles", ["The Grove", "Melrose", "Santa Monica"]),
           ("Chicago", "IL", 41.8781, -87.6298, "America/Chicago", ["Michigan Avenue", "Wicker Park"]),
           ("Houston", "TX", 29.7604, -95.3698, "America/Chicago", ["Galleria", "Midtown"]),
           ("Atlanta", "GA", 33.7490, -84.3880, "America/New_York", ["Lenox", "West End"]),
           ("Seattle", "WA", 47.6062, -122.3321, "America/Los_Angeles", ["Pike Street", "University Village"]),
           ("Boston", "MA", 42.3601, -71.0589, "America/New_York", ["Newbury Street", "Seaport"]),
           ("Miami", "FL", 25.7617, -80.1918, "America/New_York", ["Lincoln Road", "Brickell"]),
           ("Denver", "CO", 39.7392, -104.9903, "America/Denver", ["Cherry Creek", "LoDo"]),
           ("Philadelphia", "PA", 39.9526, -75.1652, "America/New_York", ["Walnut Street", "Center City"]),
           ("Portland", "OR", 45.5152, -122.6784, "America/Los_Angeles", ["Pearl District"]),
           ("Dallas", "TX", 32.7767, -96.7970, "America/Chicago", ["NorthPark", "Deep Ellum"])],
    "GB": [("London", "Greater London", 51.5145, -0.1440, "Europe/London", ["Oxford Street", "Covent Garden", "Shoreditch", "Westfield Stratford"]),
           ("Manchester", "Greater Manchester", 53.4808, -2.2426, "Europe/London", ["Market Street", "Trafford"]),
           ("Birmingham", "West Midlands", 52.4862, -1.8904, "Europe/London", ["Bullring"]),
           ("Glasgow", "Scotland", 55.8642, -4.2518, "Europe/London", ["Buchanan Street"]),
           ("Leeds", "West Yorkshire", 53.8008, -1.5491, "Europe/London", ["Briggate"]),
           ("Liverpool", "Merseyside", 53.4084, -2.9916, "Europe/London", ["Church Street"]),
           ("Bristol", "Bristol", 51.4545, -2.5879, "Europe/London", ["Cabot Circus"]),
           ("Edinburgh", "Scotland", 55.9533, -3.1883, "Europe/London", ["Princes Street"])],
    "JP": [("Tokyo", "Tokyo", 35.6702, 139.7027, "Asia/Tokyo", ["Harajuku", "Shibuya", "Ginza", "Shinjuku"]),
           ("Osaka", "Osaka", 34.6937, 135.5023, "Asia/Tokyo", ["Shinsaibashi", "Umeda"]),
           ("Yokohama", "Kanagawa", 35.4437, 139.6380, "Asia/Tokyo", ["Minato Mirai"]),
           ("Nagoya", "Aichi", 35.1815, 136.9066, "Asia/Tokyo", ["Sakae"]),
           ("Fukuoka", "Fukuoka", 33.5904, 130.4017, "Asia/Tokyo", ["Tenjin"]),
           ("Sapporo", "Hokkaido", 43.0618, 141.3545, "Asia/Tokyo", ["Odori"]),
           ("Kyoto", "Kyoto", 35.0116, 135.7681, "Asia/Tokyo", ["Shijo"])],
    "BR": [("São Paulo", "SP", -23.5614, -46.6559, "America/Sao_Paulo", ["Avenida Paulista", "Rua Oscar Freire", "Pinheiros"]),
           ("Rio de Janeiro", "RJ", -22.9068, -43.1729, "America/Sao_Paulo", ["Ipanema", "Barra da Tijuca"]),
           ("Belo Horizonte", "MG", -19.9167, -43.9345, "America/Sao_Paulo", ["Savassi"]),
           ("Brasília", "DF", -15.7975, -47.8919, "America/Sao_Paulo", ["Asa Sul"]),
           ("Curitiba", "PR", -25.4284, -49.2733, "America/Sao_Paulo", ["Batel"]),
           ("Porto Alegre", "RS", -30.0346, -51.2177, "America/Sao_Paulo", ["Moinhos de Vento"]),
           ("Recife", "PE", -8.0476, -34.8770, "America/Recife", ["Boa Viagem"])],
    "AU": [("Sydney", "NSW", -33.8688, 151.2093, "Australia/Sydney", ["Pitt Street", "Bondi Junction", "Parramatta"]),
           ("Melbourne", "VIC", -37.8136, 144.9631, "Australia/Melbourne", ["Bourke Street", "Chadstone"]),
           ("Brisbane", "QLD", -27.4698, 153.0251, "Australia/Brisbane", ["Queen Street"]),
           ("Perth", "WA", -31.9505, 115.8605, "Australia/Perth", ["Murray Street"]),
           ("Adelaide", "SA", -34.9285, 138.6007, "Australia/Adelaide", ["Rundle Mall"]),
           ("Gold Coast", "QLD", -28.0167, 153.4000, "Australia/Brisbane", ["Pacific Fair"])],
    "DE": [("Berlin", "Berlin", 52.5200, 13.4050, "Europe/Berlin", ["Kurfürstendamm", "Alexanderplatz", "Mitte"]),
           ("Munich", "Bavaria", 48.1351, 11.5820, "Europe/Berlin", ["Kaufingerstraße"]),
           ("Hamburg", "Hamburg", 53.5511, 9.9937, "Europe/Berlin", ["Mönckebergstraße", "Schanzenviertel"]),
           ("Frankfurt", "Hesse", 50.1109, 8.6821, "Europe/Berlin", ["Zeil"]),
           ("Cologne", "North Rhine-Westphalia", 50.9375, 6.9603, "Europe/Berlin", ["Schildergasse"]),
           ("Düsseldorf", "North Rhine-Westphalia", 51.2277, 6.7735, "Europe/Berlin", ["Königsallee"])],
    "FR": [("Paris", "Île-de-France", 48.8698, 2.3075, "Europe/Paris", ["Champs-Élysées", "Le Marais", "Les Halles"]),
           ("Lyon", "Auvergne-Rhône-Alpes", 45.7640, 4.8357, "Europe/Paris", ["Rue de la République"]),
           ("Marseille", "Provence-Alpes-Côte d'Azur", 43.2965, 5.3698, "Europe/Paris", ["Terrasses du Port"]),
           ("Bordeaux", "Nouvelle-Aquitaine", 44.8378, -0.5792, "Europe/Paris", ["Rue Sainte-Catherine"]),
           ("Lille", "Hauts-de-France", 50.6292, 3.0573, "Europe/Paris", ["Euralille"]),
           ("Toulouse", "Occitanie", 43.6047, 1.4442, "Europe/Paris", ["Capitole"])],
    "CA": [("Toronto", "ON", 43.6532, -79.3832, "America/Toronto", ["Eaton Centre", "Queen Street West", "Yorkdale"]),
           ("Vancouver", "BC", 49.2827, -123.1207, "America/Vancouver", ["Robson Street", "Metrotown"]),
           ("Montreal", "QC", 45.5019, -73.5674, "America/Toronto", ["Rue Sainte-Catherine"]),
           ("Calgary", "AB", 51.0447, -114.0719, "America/Edmonton", ["Chinook Centre"]),
           ("Ottawa", "ON", 45.4215, -75.6972, "America/Toronto", ["Rideau Centre"])],
    "CN": [("Shanghai", "Shanghai", 31.2304, 121.4737, "Asia/Shanghai", ["Nanjing Road", "Huaihai Road", "Xintiandi"]),
           ("Beijing", "Beijing", 39.9042, 116.4074, "Asia/Shanghai", ["Sanlitun", "Wangfujing"]),
           ("Guangzhou", "Guangdong", 23.1291, 113.2644, "Asia/Shanghai", ["Tianhe"]),
           ("Shenzhen", "Guangdong", 22.5431, 114.0579, "Asia/Shanghai", ["Futian"]),
           ("Chengdu", "Sichuan", 30.5728, 104.0668, "Asia/Shanghai", ["Chunxi Road"]),
           ("Hangzhou", "Zhejiang", 30.2741, 120.1551, "Asia/Shanghai", ["Hubin"])],
    "KR": [("Seoul", "Seoul", 37.5563, 126.9236, "Asia/Seoul", ["Myeongdong", "Hongdae", "Gangnam"]),
           ("Busan", "Busan", 35.1796, 129.0756, "Asia/Seoul", ["Seomyeon", "Haeundae"]),
           ("Incheon", "Incheon", 37.4563, 126.7052, "Asia/Seoul", ["Songdo"]),
           ("Daegu", "Daegu", 35.8714, 128.6014, "Asia/Seoul", ["Dongseongno"]),
           ("Daejeon", "Daejeon", 36.3504, 127.3845, "Asia/Seoul", ["Dunsan"])],
}

# Outlet towns for factory stores: market -> [(town, state, lat, lon, tz)]
OUTLET_TOWNS = {
    "US": [("Orlando", "FL", 28.4446, -81.4501, "America/New_York"), ("Woodbury", "NY", 41.3134, -74.1182, "America/New_York"),
           ("Cabazon", "CA", 33.9175, -116.7870, "America/Los_Angeles"), ("San Marcos", "TX", 29.8278, -97.9817, "America/Chicago")],
    "GB": [("Bicester", "Oxfordshire", 51.8916, -1.1534, "Europe/London"), ("Cheshire Oaks", "Cheshire", 53.2667, -2.8833, "Europe/London")],
    "JP": [("Gotemba", "Shizuoka", 35.3086, 138.9653, "Asia/Tokyo"), ("Rinku", "Osaka", 34.4097, 135.2967, "Asia/Tokyo")],
    "BR": [("Itupeva", "SP", -23.1531, -47.0578, "America/Sao_Paulo")],
    "AU": [("Essendon Fields", "VIC", -37.7286, 144.9026, "Australia/Melbourne"), ("Homebush", "NSW", -33.8650, 151.0800, "Australia/Sydney")],
    "DE": [("Metzingen", "Baden-Württemberg", 48.5367, 9.2858, "Europe/Berlin"), ("Wustermark", "Brandenburg", 52.5500, 12.9500, "Europe/Berlin")],
    "FR": [("Serris", "Île-de-France", 48.8540, 2.7840, "Europe/Paris"), ("Roubaix", "Hauts-de-France", 50.6942, 3.1746, "Europe/Paris")],
    "CA": [("Halton Hills", "ON", 43.6300, -79.9500, "America/Toronto")],
    "CN": [("Kunshan", "Jiangsu", 31.3858, 120.9806, "Asia/Shanghai")],
    "KR": [("Yeoju", "Gyeonggi", 37.2983, 127.6375, "Asia/Seoul"), ("Gimhae", "Gyeongnam", 35.2285, 128.8894, "Asia/Seoul")],
}

# Distribution centres: (dc_id, market, city, state, lat, lon, tz, priority); priority 1 = primary DC for the market
DCS = [
    ("DC-US-MEM", "US", "Memphis", "TN", 35.1495, -90.0490, "America/Chicago", 1),
    ("DC-GB-DAV", "GB", "Daventry", "Northamptonshire", 52.2565, -1.1626, "Europe/London", 1),
    ("DC-JP-ICH", "JP", "Ichikawa", "Chiba", 35.7219, 139.9310, "Asia/Tokyo", 1),
    ("DC-BR-CAJ", "BR", "Cajamar", "SP", -23.3550, -46.8769, "America/Sao_Paulo", 1),
    ("DC-AU-TRU", "AU", "Truganina", "VIC", -37.8167, 144.7333, "Australia/Melbourne", 1),
    ("DC-DE-ERF", "DE", "Erfurt", "Thuringia", 50.9848, 11.0299, "Europe/Berlin", 1),
    ("DC-FR-LIE", "FR", "Lieusaint", "Île-de-France", 48.6333, 2.5500, "Europe/Paris", 1),
    ("DC-CA-MIS", "CA", "Mississauga", "ON", 43.5890, -79.6441, "America/Toronto", 1),
    ("DC-CN-TAI", "CN", "Taicang", "Jiangsu", 31.4597, 121.1302, "Asia/Shanghai", 1),
    ("DC-KR-ICN", "KR", "Incheon", "Incheon", 37.4563, 126.7052, "Asia/Seoul", 1),
    ("DC-US-ONT", "US", "Ontario", "CA", 34.0633, -117.6509, "America/Los_Angeles", 2),
    ("DC-US-ALT", "US", "Allentown", "PA", 40.6084, -75.4902, "America/New_York", 2),
]

# Product families: (category, division, gender, base model names, msrp tiers USD, size run, popularity)
FAMILIES = [
    ("RUNNING", "FOOTWEAR", "M", ["AeroGlide", "Pacer", "Velo Stride", "Tempo Flux", "Cloudline"], [110, 130, 160, 180], "MENS_FW", 1.30),
    ("RUNNING", "FOOTWEAR", "W", ["AeroGlide", "Pacer", "Velo Stride", "Tempo Flux", "Cloudline"], [110, 130, 160, 180], "WOMENS_FW", 1.15),
    ("TRAIL", "FOOTWEAR", "M", ["Ridgeline", "Summit Trek", "Scree Runner"], [120, 140, 165], "MENS_FW", 0.55),
    ("TRAIL", "FOOTWEAR", "W", ["Ridgeline", "Summit Trek"], [120, 140], "WOMENS_FW", 0.45),
    ("BASKETBALL", "FOOTWEAR", "M", ["Court Vortex", "Skyrise", "Rim Elite", "Baseline Pro"], [100, 125, 150, 190], "MENS_FW", 0.95),
    ("LIFESTYLE", "FOOTWEAR", "M", ["Heritage 82", "Metro Classic", "Cortex Low", "Retro Runner"], [85, 100, 115, 130], "MENS_FW", 1.25),
    ("LIFESTYLE", "FOOTWEAR", "W", ["Heritage 82", "Metro Classic", "Cortex Low", "Platform Muse"], [85, 100, 115, 130], "WOMENS_FW", 1.20),
    ("TRAINING", "FOOTWEAR", "M", ["Flexform TR", "Grit Trainer", "Lift Core"], [90, 110, 130], "MENS_FW", 0.60),
    ("TRAINING", "FOOTWEAR", "W", ["Flexform TR", "Grit Trainer", "Studio Flow"], [85, 100, 120], "WOMENS_FW", 0.65),
    ("FOOTBALL", "FOOTWEAR", "M", ["Strike Elite FG", "Apex Control FG", "Velocity Club FG"], [70, 150, 260], "MENS_FW", 0.50),
    ("APPAREL", "APPAREL", "M", ["AirLite Run Tee", "StormShield Jacket", "CoreFleece Hoodie", "Stride Short 7in", "Tempo Tight"], [35, 45, 70, 90, 130], "APPAREL", 0.90),
    ("APPAREL", "APPAREL", "W", ["AirLite Run Tank", "StormShield Jacket", "CoreFleece Crew", "Stride Short 3in", "Sculpt Legging"], [35, 45, 70, 80, 110], "APPAREL", 0.95),
]

STYLE_PREFIXES = ["HV", "FV", "DV", "FD", "DX", "HQ", "FJ", "IB", "FN", "HF"]

COLORWAYS = [
    ("001", "Black/White"), ("100", "White/Black"), ("002", "Black/Anthracite-Cool Grey"), ("101", "White/Summit White-Pure Platinum"),
    ("400", "Midnight Navy/White"), ("600", "University Red/Black"), ("300", "Volt/Black"), ("700", "Laser Orange/Sail"),
    ("401", "Racer Blue/White-Black"), ("500", "Hyper Violet/Pink Blast"), ("200", "Light Bone/Sail-Gum"), ("003", "Iron Grey/Smoke Grey"),
    ("301", "Electric Green/Deep Jungle"), ("601", "Team Crimson/Phantom"), ("402", "Glacier Blue/Ice"), ("201", "Khaki/Sequoia"),
]

SIZE_RUNS = {
    "MENS_FW": ["6", "6.5", "7", "7.5", "8", "8.5", "9", "9.5", "10", "10.5", "11", "11.5", "12", "13", "14"],
    "WOMENS_FW": ["5", "5.5", "6", "6.5", "7", "7.5", "8", "8.5", "9", "9.5", "10", "11"],
    "APPAREL": ["XS", "S", "M", "L", "XL", "XXL"],
}
SIZE_CURVE_CENTER = {"MENS_FW": (10.0, 1.6), "WOMENS_FW": (7.75, 1.4)}
APPAREL_CURVE = {"XS": 0.06, "S": 0.18, "M": 0.30, "L": 0.26, "XL": 0.14, "XXL": 0.06}

RETURN_RATE = {"FOOTWEAR": 0.055, "APPAREL": 0.085}
UNIT_COST_RATIO = {"FOOTWEAR": 0.36, "APPAREL": 0.30}

WHOLESALE_PARTNERS = [
    ("Summit Sports Group", "US", "KEY"), ("Swiftline Athletics", "US", "KEY"), ("Courtside Collective", "US", "CORE"),
    ("Harbor & Field Sports", "GB", "KEY"), ("Northline Outfitters", "GB", "CORE"), ("Stride & Co", "GB", "CORE"),
    ("Sakura Run Depot", "JP", "KEY"), ("Kanto Athletic House", "JP", "CORE"), ("Tropico Esportes", "BR", "KEY"),
    ("Arena Sul Esportes", "BR", "CORE"), ("Southern Cross Sport", "AU", "KEY"), ("Coastline Athletic", "AU", "CORE"),
    ("Rhein Sporthaus", "DE", "KEY"), ("Alpenlauf Sport", "DE", "CORE"), ("Maison du Sport Lumière", "FR", "KEY"),
    ("Maple Leaf Athletics", "CA", "KEY"), ("Jade River Sports", "CN", "KEY"), ("Hanbit Sports", "KR", "KEY"),
]

VENDORS = [
    ("VN-DNF", "Dong Nai Footwear Co. (synthetic vendor)", "VN", 38),
    ("VN-BDS", "Binh Duong Shoe Works (synthetic vendor)", "VN", 40),
    ("ID-TGR", "Tangerang Footwear Industries (synthetic vendor)", "ID", 42),
    ("CN-PUT", "Putian Athletic Manufacturing (synthetic vendor)", "CN", 30),
    ("KH-PPA", "Phnom Penh Apparel Ltd. (synthetic vendor)", "KH", 45),
    ("BD-DHA", "Dhaka Knit Partners (synthetic vendor)", "BD", 48),
]

# People (all invented). (person_id, display name, title, market or GLOBAL)
PEOPLE = [
    ("dortiz", "Daniela Ortiz", "Regional Marketplace Director, North America", "US"),
    ("pnair", "Priya Nair", "Senior Merchandise Planner, Running", "US"),
    ("mlee", "Marcus Lee", "Wholesale Account Lead, North America", "US"),
    ("ohaddad", "Omar Haddad", "Finance Partner, Marketplace", "GLOBAL"),
    ("fbrandt", "Fiona Brandt", "Marketplace Director, Europe", "GB"),
    ("lweber", "Lukas Weber", "Merchandise Planner, Europe", "DE"),
    ("erossi", "Elena Rossi", "Wholesale Account Lead, Europe", "GB"),
    ("mtan", "Mei Tan", "Marketplace Director, Asia Pacific", "JP"),
    ("asato", "Aiko Sato", "Merchandise Planner, Japan", "JP"),
    ("gkim", "Grace Kim", "Inventory Analyst, Asia Pacific", "AU"),
    ("slima", "Sofia Lima", "Merchandise Planner, Brazil", "BR"),
    ("lcarter", "Liam Carter", "Merchandise Planner, Australia", "AU"),
    ("jmartin", "Julien Martin", "Merchandise Planner, France", "FR"),
    ("cwong", "Chloe Wong", "Merchandise Planner, Canada", "CA"),
    ("hzhang", "Hao Zhang", "Merchandise Planner, Greater China", "CN"),
    ("jpark", "Jiwoo Park", "Merchandise Planner, Korea", "KR"),
]
# Invented store-manager names (one manager per store; picked deterministically from the seed)
STORE_MANAGER_FIRST = ["Alex", "Jordan", "Sam", "Taylor", "Morgan", "Casey", "Riley", "Jamie", "Avery", "Quinn",
                       "Rowan", "Harper", "Emery", "Dakota", "Reese", "Skyler", "Parker", "Sage", "Remy", "Noa"]
STORE_MANAGER_LAST = ["Bennett", "Okafor", "Silva", "Novak", "Ramos", "Lindqvist", "Moreau", "Tanaka", "Haddad",
                      "Kowalski", "Fischer", "Mensah", "Duarte", "Ivanova", "Park", "Rossi", "Nakamura", "Byrne"]
MARKET_DIRECTOR = {"US": "dortiz", "CA": "dortiz", "GB": "fbrandt", "DE": "fbrandt", "FR": "fbrandt",
                   "JP": "mtan", "AU": "mtan", "CN": "mtan", "KR": "mtan", "BR": "dortiz"}
MARKET_PLANNER = {"US": "pnair", "GB": "lweber", "DE": "lweber", "FR": "jmartin", "JP": "asato", "BR": "slima",
                  "AU": "lcarter", "CA": "cwong", "CN": "hzhang", "KR": "jpark"}
WHOLESALE_LEAD = {"US": "mlee", "CA": "mlee", "BR": "mlee", "GB": "erossi", "DE": "erossi", "FR": "erossi",
                  "JP": "gkim", "AU": "gkim", "CN": "gkim", "KR": "gkim"}

# Regional events that drive demand spikes: (market, name, month, day, duration_days, categories, uplift)
REGIONAL_EVENTS = [
    ("US", "Big Apple City Marathon Expo", 11, 1, 3, ["RUNNING"], 1.6),
    ("US", "Back-to-School Season", 8, 10, 21, ["LIFESTYLE", "BASKETBALL", "APPAREL"], 1.25),
    ("US", "Black Friday Weekend", 11, 27, 4, None, 1.9),
    ("GB", "Thames Half Marathon", 10, 12, 2, ["RUNNING"], 1.4),
    ("GB", "Boxing Day Sales", 12, 26, 3, None, 1.8),
    ("JP", "Tokyo City Run Expo", 10, 24, 1, ["RUNNING"], 2.2),
    ("JP", "Golden Week", 4, 29, 7, None, 1.3),
    ("BR", "Carnaval", 2, 14, 5, ["LIFESTYLE", "APPAREL"], 1.3),
    ("BR", "São Paulo Night Run", 12, 31, 1, ["RUNNING"], 1.5),
    ("AU", "Harbour City Fun Run", 8, 9, 2, ["RUNNING"], 1.4),
    ("AU", "Boxing Day Sales", 12, 26, 3, None, 1.7),
    ("DE", "Spree River Marathon", 9, 21, 2, ["RUNNING"], 1.5),
    ("FR", "Les Soldes d'hiver", 1, 8, 14, None, 1.5),
    ("CA", "Boxing Week", 12, 26, 7, None, 1.6),
    ("CN", "Singles Day", 11, 11, 2, None, 2.4),
    ("KR", "Chuseok", 10, 5, 3, None, 1.2),
]

MONTH_FACTOR_N = [0.85, 0.80, 0.95, 0.97, 1.00, 1.00, 0.95, 1.12, 1.02, 0.96, 1.18, 1.35]
WEEKDAY_FACTOR = [0.82, 0.80, 0.85, 0.92, 1.12, 1.38, 1.18]   # Mon..Sun
FORMAT_FACTOR = {"FLAGSHIP": 2.4, "BRAND": 1.0, "COMMUNITY": 0.55, "FACTORY": 1.3, "ECOM": 4.2}

BANNED_REAL_BRANDS = ["nike", "jordan", "adidas", "puma", "reebok", "asics", "new balance", "under armour",
                      "foot locker", "jd sports", "dick's", "pegasus", "vaporfly", "air max", "dri-fit", "swoosh"]

In [ ]:
# Parameters: size profile + overrides from the widgets (blank widget = profile default)
PROFILES: dict[str, dict] = {
    # unit tests
    "tiny": dict(n_markets=5, stores_per_market=3, n_styles=12, history_days=75),
    # quick demo / laptop
    "small": dict(n_markets=5, stores_per_market=6, n_styles=24, history_days=120),
    # default for Databricks / Fabric demos
    "demo": dict(n_markets=6, stores_per_market=15, n_styles=60, history_days=365),
    # stress test (Spark sizing, partitioning)
    "large": dict(n_markets=10, stores_per_market=40, n_styles=150, history_days=730, demand_scale=1.5),
}


@dataclass(frozen=True)
class SynthParams:
    seed: int = 20261106
    as_of: str = "2026-10-31"          # "today" in the synthetic world; batch history ends the day before
    history_days: int = 120
    n_markets: int = 5                 # 5..10; US GB JP BR AU always first (they carry the scenario)
    stores_per_market: int = 6         # physical stores per market (+1 e-commerce node per market)
    n_styles: int = 24                 # 12..200
    demand_scale: float = 1.0          # multiplies unit demand
    dq_defect_rate: float = 0.0005     # share of fact rows that also get a defective copy
    brand: str = "Kinetiq"
    profile: str = "custom"

    @property
    def as_of_date(self) -> date:
        return date.fromisoformat(self.as_of)

    @property
    def history_start(self) -> date:
        return self.as_of_date - timedelta(days=self.history_days)

    @property
    def hero_launch(self) -> date:
        return self.as_of_date - timedelta(days=55)

    def validate(self) -> "SynthParams":
        if not 5 <= self.n_markets <= 10:
            raise ValueError("n_markets must be 5..10")
        if not 3 <= self.stores_per_market <= 60:
            raise ValueError("stores_per_market must be 3..60")
        if not 12 <= self.n_styles <= 200:
            raise ValueError("n_styles must be 12..200")
        if not 75 <= self.history_days <= 1095:
            raise ValueError("history_days must be 75..1095 (hero launch is as_of - 55 days)")
        if self.demand_scale <= 0:
            raise ValueError("demand_scale must be > 0")
        if not 0 <= self.dq_defect_rate <= 0.05:
            raise ValueError("dq_defect_rate must be 0..0.05")
        date.fromisoformat(self.as_of)
        return self

    def to_dict(self) -> dict:
        return asdict(self)

    def to_json(self) -> str:
        return json.dumps(self.to_dict(), sort_keys=True)

    def fingerprint(self) -> str:
        return hashlib.sha256(self.to_json().encode()).hexdigest()[:16]


_TYPES = {f.name: f.type for f in fields(SynthParams)}


def make_params(profile: str | None = "small", **overrides) -> SynthParams:
    """Profile defaults plus overrides. Empty-string / None overrides are ignored, so notebook widgets can be
    passed straight through."""
    values: dict = {}
    if profile:
        if profile not in PROFILES:
            raise ValueError(f"unknown profile {profile!r}; choose one of {sorted(PROFILES)}")
        values.update(PROFILES[profile])
        values["profile"] = profile
    for k, v in overrides.items():
        if v is None or (isinstance(v, str) and v.strip() == ""):
            continue
        if k not in _TYPES:
            raise ValueError(f"unknown parameter {k!r}")
        t = _TYPES[k]
        values[k] = int(v) if t in ("int", int) else float(v) if t in ("float", float) else str(v)
    return SynthParams(**values).validate()


params = make_params(dbutils.widgets.get("profile"),
                     **{k: dbutils.widgets.get(k) for k in ["seed", "as_of", "history_days", "n_markets", "stores_per_market", "n_styles", "demand_scale", "dq_defect_rate"]})
print(params.to_json())
print("fingerprint:", params.fingerprint())

In [ ]:
# World: master data and the demo scenario (deterministic from the parameters)
SYNTHETIC_LABEL = "SYNTHETIC"
HERO_STYLE = "HM9443"
HERO_COLORS = ("001", "100")
HERO_SURGE_SIZES = ("9", "9.5", "10", "10.5", "11")
STORE_LANE_MAX_KM = 1500        # same-market store-to-store truck lanes only between stores this close
SCENARIO_MARKETS = ("US", "GB", "JP", "BR", "AU")


def rng_for(seed: int, *key) -> np.random.Generator:
    """Independent, order-free RNG stream per key (stable across processes and platforms)."""
    h = zlib.crc32("|".join(map(str, key)).encode())
    return np.random.default_rng([seed, h])


def gtin13(body12: str) -> str:
    digits = [int(c) for c in body12]
    s = sum(d * (3 if i % 2 else 1) for i, d in enumerate(digits))
    return body12 + str((10 - s % 10) % 10)


def round_price(usd: float, fx: float, premium: float, rule: str) -> float:
    v = usd * fx * premium
    if rule == "x.99":
        return float(math.ceil(v) - 0.01)
    if rule == "x9":
        return float(math.ceil(v / 10) * 10 - 1)
    step = int(rule)
    return float(math.ceil(v / step) * step)


@dataclass
class World:
    params: SynthParams
    markets: pd.DataFrame
    locations: pd.DataFrame
    styles: pd.DataFrame
    skus: pd.DataFrame
    assortment: pd.DataFrame          # location_id x style_color carried
    prices: pd.DataFrame
    partners: pd.DataFrame
    lanes: pd.DataFrame
    people: pd.DataFrame
    owners: pd.DataFrame
    vendors: pd.DataFrame
    events: pd.DataFrame
    store_managers: pd.DataFrame = field(default_factory=pd.DataFrame)   # location_id -> person_id (STORE only)
    scenario: dict = field(default_factory=dict)

    @property
    def market_ids(self) -> list[str]:
        return self.markets.market_id.tolist()

    def selling_locations(self, market_id: str | None = None) -> pd.DataFrame:
        loc = self.locations[self.locations.location_type.isin(["STORE", "ECOM"])]
        return loc if market_id is None else loc[loc.market_id == market_id]


# ---------------------------------------------------------------- markets & locations
def _markets(p: SynthParams) -> pd.DataFrame:
    rows = MARKETS[: p.n_markets]
    return pd.DataFrame(rows, columns=["market_id", "market_name", "region", "currency", "fx_usd_to_local",
                                       "price_premium", "hemisphere", "timezone", "price_rounding"])


def _formats(n: int) -> list[str]:
    n_flag = 1 if n >= 5 else 0
    n_fac = max(1, round(n * 0.15))
    n_com = max(1, round(n * 0.15)) if n >= 4 else 0
    n_brand = n - n_flag - n_fac - n_com
    return ["FLAGSHIP"] * n_flag + ["BRAND"] * n_brand + ["COMMUNITY"] * n_com + ["FACTORY"] * n_fac


def _dist(a, b) -> float:
    return math.hypot(a[0] - b[0], (a[1] - b[1]) * math.cos(math.radians(a[0])))


def _locations(p: SynthParams, markets: pd.DataFrame) -> pd.DataFrame:
    rows = []
    brand = p.brand
    dcs = [d for d in DCS if d[1] in set(markets.market_id) and (d[7] == 1 or p.stores_per_market >= 10)]
    for d in dcs:
        rows.append(dict(location_id=d[0], location_name=f"{brand} Distribution Center {d[2]}", location_type="DC",
                         format="DC", market_id=d[1], city=d[2], state_province=d[3], latitude=d[4], longitude=d[5],
                         timezone=d[6], serving_dc_id=None, opened_date=date(2008 + len(rows) % 9, 3, 1),
                         size_factor=0.0))
    for m in markets.itertuples():
        r = rng_for(p.seed, "loc", m.market_id)
        cities = CITIES[m.market_id]
        outlets = OUTLET_TOWNS[m.market_id]
        slots = [(c, d) for c in cities for d in c[5]]           # (city, district)
        # weight: first districts of each city first, so stores spread across cities before doubling up
        slots.sort(key=lambda s: s[0][5].index(s[1]))
        used = 0
        fac_i = 0
        mdc = [d for d in dcs if d[1] == m.market_id]
        for i, fmt in enumerate(_formats(p.stores_per_market)):
            store_no = 1001 + i
            lid = f"ST-{m.market_id}-{store_no}"
            if fmt == "FACTORY":
                town = outlets[fac_i % len(outlets)]
                fac_i += 1
                suffix = "" if fac_i <= len(outlets) else f" {fac_i // len(outlets) + 1}"
                city, state, lat, lon, tz = town
                name = f"{brand} Factory Store {city}{suffix}"
            else:
                (c, district) = slots[used % len(slots)]
                lap = used // len(slots)
                used += 1
                city, state, lat, lon, tz = c[0], c[1], c[2], c[3], c[4]
                lat += float(r.normal(0, 0.02))
                lon += float(r.normal(0, 0.02))
                extra = f" {lap + 1}" if lap else ""
                if fmt == "FLAGSHIP":
                    name = f"{brand} Flagship {city} {district}"
                elif fmt == "COMMUNITY":
                    name = f"{brand} Community Store {city} {district}{extra}"
                else:
                    name = f"{brand} {city} {district}{extra}"
            near = min(mdc, key=lambda d: _dist((lat, lon), (d[4], d[5])))
            rows.append(dict(location_id=lid, location_name=name, location_type="STORE", format=fmt,
                             market_id=m.market_id, city=city, state_province=state, latitude=round(lat, 5),
                             longitude=round(lon, 5), timezone=tz, serving_dc_id=near[0],
                             opened_date=date(int(r.integers(2004, 2025)), int(r.integers(1, 13)), 1),
                             size_factor=float(np.clip(r.lognormal(0, 0.28), 0.55, 1.9))))
        primary = mdc[0]
        rows.append(dict(location_id=f"EC-{m.market_id}", location_name=f"{brand}.com {m.market_name}",
                         location_type="ECOM", format="ECOM", market_id=m.market_id, city=primary[2],
                         state_province=primary[3], latitude=primary[4], longitude=primary[5], timezone=m.timezone,
                         serving_dc_id=primary[0], opened_date=date(2012, 6, 1), size_factor=1.0))
    df = pd.DataFrame(rows)
    df["record_label"] = SYNTHETIC_LABEL
    return df


# ---------------------------------------------------------------- products
def _styles(p: SynthParams) -> pd.DataFrame:
    r = rng_for(p.seed, "styles")
    as_of = p.as_of_date
    rows = [dict(style_id=HERO_STYLE, style_name="Men's AeroGlide 5", model="AeroGlide", category="RUNNING",
                 division="FOOTWEAR", gender="M", size_run="MENS_FW", msrp_usd=140.0,
                 launch_date=p.hero_launch, popularity=2.4, n_colors=2)]
    used_ids = {HERO_STYLE}
    used_names = {"Men's AeroGlide 5"}
    fam_w = np.array([f[6] for f in FAMILIES])
    fam_w = fam_w / fam_w.sum()
    while len(rows) < p.n_styles:
        fi = int(r.choice(len(FAMILIES), p=fam_w))
        cat, div, gender, models, tiers, run, pop = FAMILIES[fi]
        mi = int(r.integers(len(models)))
        model = models[mi]
        version = int(r.integers(2, 13)) if div == "FOOTWEAR" and not model[-1].isdigit() and "FG" not in model else None
        g = "Men's" if gender == "M" else "Women's"
        name = f"{g} {model}" + (f" {version}" if version else "")
        if name in used_names:
            name = f"{name} {['SE', 'Premium', 'GTX', 'Next Nature', 'Wide'][len(used_names) % 5]}"
            if name in used_names:
                continue
        while True:
            sid = f"{STYLE_PREFIXES[int(r.integers(len(STYLE_PREFIXES)))]}{int(r.integers(1000, 10000))}"
            if sid not in used_ids:
                break
        used_ids.add(sid)
        used_names.add(name)
        tier = tiers[min(mi, len(tiers) - 1)] if len(tiers) > mi else tiers[-1]
        # launch: mostly established product, ~20% launched inside the last 6 months
        age = int(r.integers(20, 180)) if r.random() < 0.2 else int(r.integers(180, 760))
        rows.append(dict(style_id=sid, style_name=name, model=model, category=cat, division=div, gender=gender,
                         size_run=run, msrp_usd=float(tier), launch_date=as_of - timedelta(days=age),
                         popularity=float(pop * r.lognormal(0, 0.45)),
                         n_colors=int(r.integers(3, 5)) if div == "APPAREL" else int(r.integers(2, 5))))
    df = pd.DataFrame(rows)
    age = (pd.Timestamp(as_of) - pd.to_datetime(df.launch_date)).dt.days
    df["lifecycle"] = np.where(age < 60, "LAUNCH", np.where(age > 540, "MARKDOWN", "CORE"))
    df["unit_cost_usd"] = (df.msrp_usd * df.division.map(UNIT_COST_RATIO)).round(2)
    df["record_label"] = SYNTHETIC_LABEL
    return df


def _size_weights(run: str) -> dict[str, float]:
    sizes = SIZE_RUNS[run]
    if run == "APPAREL":
        w = np.array([APPAREL_CURVE[s] for s in sizes])
    else:
        mu, sd = SIZE_CURVE_CENTER[run]
        w = np.array([math.exp(-0.5 * ((float(s) - mu) / sd) ** 2) for s in sizes])
    w = w / w.sum()
    return dict(zip(sizes, w))


def _skus(p: SynthParams, styles: pd.DataFrame) -> pd.DataFrame:
    rows = []
    seq = 0
    for s in styles.itertuples():
        r = rng_for(p.seed, "colors", s.style_id)
        if s.style_id == HERO_STYLE:
            colors = [c for c in COLORWAYS if c[0] in HERO_COLORS]
            shares = [0.62, 0.38]
        else:
            idx = sorted(r.choice(len(COLORWAYS), s.n_colors, replace=False))
            colors = [COLORWAYS[i] for i in idx]
            shares = list(r.dirichlet(np.full(len(colors), 2.5)))
        sw = _size_weights(s.size_run)
        for (code, cname), cshare in zip(colors, shares):
            for size, w in sw.items():
                seq += 1
                rows.append(dict(sku_id=f"{s.style_id}-{code}-{size}", style_id=s.style_id, color_code=code,
                                 color_name=cname, size=size, size_run=s.size_run,
                                 gtin=gtin13(f"029{seq:09d}"), color_share=float(cshare), size_weight=float(w)))
    df = pd.DataFrame(rows)
    df["record_label"] = SYNTHETIC_LABEL
    return df


def _assortment(p: SynthParams, locations: pd.DataFrame, styles: pd.DataFrame, skus: pd.DataFrame) -> pd.DataFrame:
    sc = skus[["style_id", "color_code"]].drop_duplicates()
    sc = sc.merge(styles[["style_id", "lifecycle", "launch_date"]], on="style_id")
    rows = []
    for loc in locations[locations.location_type.isin(["STORE", "ECOM"])].itertuples():
        r = rng_for(p.seed, "assort", loc.location_id)
        u = r.random(len(sc))
        if loc.format in ("FLAGSHIP", "ECOM"):
            keep = np.ones(len(sc), bool)
        elif loc.format == "FACTORY":
            keep = (sc.lifecycle.to_numpy() != "LAUNCH") & (u < 0.75)
        else:
            rate = 0.8 if loc.format == "BRAND" else 0.55
            keep = (u < rate) & (sc.lifecycle.to_numpy() != "MARKDOWN") | (sc.style_id.to_numpy() == HERO_STYLE)
        part = sc.loc[keep, ["style_id", "color_code"]].copy()
        part["location_id"] = loc.location_id
        rows.append(part)
    return pd.concat(rows, ignore_index=True)[["location_id", "style_id", "color_code"]]


def _prices(styles: pd.DataFrame, markets: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for s in styles.itertuples():
        for m in markets.itertuples():
            rows.append(dict(style_id=s.style_id, market_id=m.market_id, currency=m.currency,
                             full_price_local=round_price(s.msrp_usd, m.fx_usd_to_local, m.price_premium, m.price_rounding),
                             markdown_pct=0.30 if s.lifecycle == "MARKDOWN" else 0.0,
                             effective_from=min(s.launch_date, date(2025, 1, 1)) if s.lifecycle != "LAUNCH" else s.launch_date))
    df = pd.DataFrame(rows)
    df["record_label"] = SYNTHETIC_LABEL
    return df


# ---------------------------------------------------------------- partners, lanes, people
def _partners(markets: pd.DataFrame, locations: pd.DataFrame) -> pd.DataFrame:
    mk = set(markets.market_id)
    rows = []
    for i, (name, m, tier) in enumerate(WHOLESALE_PARTNERS):
        if m not in mk:
            continue
        dc = locations[(locations.market_id == m) & (locations.location_type == "DC")].location_id.iloc[0]
        rows.append(dict(partner_id=f"WA-{1001 + i}", partner_name=name, market_id=m, tier=tier, ship_from_dc_id=dc))
    df = pd.DataFrame(rows)
    df["record_label"] = SYNTHETIC_LABEL
    return df


def _lanes(p: SynthParams, locations: pd.DataFrame) -> pd.DataFrame:
    rows = []
    dcs = locations[locations.location_type == "DC"]
    for a in dcs.itertuples():
        for b in dcs.itertuples():
            if a.location_id == b.location_id:
                continue
            same = a.market_id == b.market_id
            km = _dist((a.latitude, a.longitude), (b.latitude, b.longitude)) * 111
            rows.append(dict(from_location_id=a.location_id, to_location_id=b.location_id,
                             mode="TRUCK" if same else ("AIR" if km > 6000 else "OCEAN_AIR"),
                             lead_days=int(2 + km // 900) if same else int(min(14, 5 + km // 2500)),
                             cost_per_unit_usd=round(0.35 + km * (0.00012 if same else 0.00055), 2)))
    for s in locations[locations.location_type.isin(["STORE", "ECOM"])].itertuples():
        dc = locations.set_index("location_id").loc[s.serving_dc_id]
        km = _dist((s.latitude, s.longitude), (dc.latitude, dc.longitude)) * 111
        for a, b in ((s.serving_dc_id, s.location_id), (s.location_id, s.serving_dc_id)):
            rows.append(dict(from_location_id=a, to_location_id=b, mode="TRUCK", lead_days=int(1 + km // 700),
                             cost_per_unit_usd=round(0.25 + km * 0.0002, 2)))
    # Store-to-store rebalancing lanes: same market, within a day or two by truck (UC1 store extension)
    stores = locations[locations.location_type == "STORE"]
    for a in stores.itertuples():
        for b in stores.itertuples():
            if a.location_id == b.location_id or a.market_id != b.market_id:
                continue
            km = _dist((a.latitude, a.longitude), (b.latitude, b.longitude)) * 111
            if km > STORE_LANE_MAX_KM:
                continue
            rows.append(dict(from_location_id=a.location_id, to_location_id=b.location_id, mode="TRUCK",
                             lead_days=int(1 + km // 800), cost_per_unit_usd=round(0.6 + km * 0.00025, 2)))
    df = pd.DataFrame(rows)
    df["record_label"] = SYNTHETIC_LABEL
    return df


def _people_owners(markets: pd.DataFrame, styles: pd.DataFrame):
    people = pd.DataFrame(PEOPLE, columns=["person_id", "display_name", "title", "home_market_id"])
    people["email"] = people.person_id + "@kinetiq.example"
    people["record_label"] = SYNTHETIC_LABEL
    rows = []
    cats = sorted(styles.category.unique())
    for m in markets.market_id:
        for c in cats:
            if (m, c) == ("BR", "TRAIL"):
                pass                                          # DQ-12: no accountable owner
            else:
                rows.append((c, m, "ACCOUNTABLE", MARKET_DIRECTOR[m], 5000))
            if (m, c) == ("AU", "LIFESTYLE"):
                rows.append((c, m, "ACCOUNTABLE", "gkim", 2000))  # DQ-12: two accountable owners
            rows.append((c, m, "PLANNER", MARKET_PLANNER[m], 500))
            rows.append((c, m, "WHOLESALE_LEAD", WHOLESALE_LEAD[m], 0))
            rows.append((c, m, "FINANCE", "ohaddad", 0))
    owners = pd.DataFrame(rows, columns=["category", "market_id", "role", "person_id", "approval_limit_units"])
    owners["record_label"] = SYNTHETIC_LABEL
    return people, owners


def _store_managers(p: SynthParams, locations: pd.DataFrame, people: pd.DataFrame):
    """One invented manager per store: extra ref.person rows plus the ref.store_manager mapping."""
    r = rng_for(p.seed, "store-manager")
    rows, links = [], []
    for s in locations[locations.location_type == "STORE"].itertuples():
        pid = "sm-" + s.location_id[3:].lower()                     # ST-US-1001 -> sm-us-1001
        name = f"{r.choice(STORE_MANAGER_FIRST)} {r.choice(STORE_MANAGER_LAST)}"
        rows.append(dict(person_id=pid, display_name=name, title=f"Store Manager, {s.location_name}"[:80],
                         home_market_id=s.market_id, email=f"{pid}@kinetiq.example", record_label=SYNTHETIC_LABEL))
        links.append(dict(location_id=s.location_id, person_id=pid, record_label=SYNTHETIC_LABEL))
    return pd.concat([people, pd.DataFrame(rows)], ignore_index=True), pd.DataFrame(links)


def _events(p: SynthParams, markets: pd.DataFrame) -> pd.DataFrame:
    rows = []
    mk = set(markets.market_id)
    for y in range(p.history_start.year, p.as_of_date.year + 1):
        for (m, name, mo, d, dur, cats, up) in REGIONAL_EVENTS:
            if m not in mk:
                continue
            start = date(y, mo, d)
            end = start + timedelta(days=dur - 1)
            if end < p.history_start or start > p.as_of_date + timedelta(days=30):
                continue
            rows.append(dict(event_id=f"EV-{m}-{start:%Y%m%d}", market_id=m, event_name=name, start_date=start,
                             end_date=end, categories=",".join(cats) if cats else "ALL", demand_uplift=up))
    df = pd.DataFrame(rows)
    df["record_label"] = SYNTHETIC_LABEL
    return df


def build_world(p: SynthParams) -> World:
    p.validate()
    markets = _markets(p)
    locations = _locations(p, markets)
    styles = _styles(p)
    skus = _skus(p, styles)
    people, owners = _people_owners(markets, styles)
    people, store_managers = _store_managers(p, locations, people)
    vendors = pd.DataFrame(VENDORS, columns=["vendor_id", "vendor_name", "country", "lead_days"])
    vendors["record_label"] = SYNTHETIC_LABEL
    w = World(params=p, markets=markets, locations=locations, styles=styles, skus=skus,
              assortment=_assortment(p, locations, styles, skus), prices=_prices(styles, markets),
              partners=_partners(markets, locations), lanes=_lanes(p, locations), people=people, owners=owners,
              vendors=vendors, events=_events(p, markets), store_managers=store_managers)
    w.scenario = dict(
        hero_style=HERO_STYLE, hero_colors=list(HERO_COLORS), hero_launch=p.hero_launch.isoformat(),
        surge_market="US", surge_sizes=list(HERO_SURGE_SIZES), surge_from_day=18, surge_sales=1.8, surge_intent=1.6,
        slow_market="GB", slow_sales=0.3, slow_intent=0.6,
        spike_market="JP", spike_event="Tokyo City Run Expo", spike_intent=3.2,
        stale_market="BR", stale_days=2,
        restated_date=(p.as_of_date - timedelta(days=3)).isoformat(),
        reserve_partner="WA-1003", cancel_partner="WA-1004", slipped_po="4577091802", po_slip_days=9,
        negative_stock_location="ST-AU-1002", overcommit_dc="DC-AU-TRU",
        # Store extension: the slowest hero stores in the surge market were over-allocated at launch and hold 9-12 units
        # of each surge size (DC spare first, the rest is part of the launch allocation)
        donor_store_count=3, donor_store_units=(9, 12),
    )
    return w


world = build_world(params)
print(f"markets={world.market_ids}  locations={len(world.locations)}  styles={len(world.styles)}  "
      f"skus={len(world.skus)}  assortment={len(world.assortment)}")

In [ ]:
# Expected-demand model (seasonality, weekday, lifecycle, events, scenario multipliers)
BASE_UNITS_PER_STORE_DAY = 70.0     # a typical BRAND store sells ~70 units/day across footwear + apparel


class MarketDemand:
    """Pre-computed (location x sku) base rates for one market. `rates(days)` -> array [day, loc, sku]."""

    def __init__(self, w: World, market_id: str):
        self.w = w
        self.market_id = market_id
        p = w.params
        m = w.markets.set_index("market_id").loc[market_id]
        self.hemisphere = m.hemisphere
        self.locs = w.selling_locations(market_id).reset_index(drop=True)
        skus = w.skus.merge(w.styles[["style_id", "category", "division", "launch_date", "lifecycle", "popularity"]],
                            on="style_id")
        self.skus = skus.reset_index(drop=True)
        n_l, n_s = len(self.locs), len(self.skus)

        carried = np.zeros((n_l, n_s), bool)
        key = self.skus.style_id + "|" + self.skus.color_code
        for i, loc in enumerate(self.locs.location_id):
            a = w.assortment[w.assortment.location_id == loc]
            carried[i] = key.isin(set(a.style_id + "|" + a.color_code)).to_numpy()
        self.carried = carried

        style_share = self.skus.popularity.to_numpy() / w.styles.popularity.sum()
        sku_share = style_share * self.skus.color_share.to_numpy() * self.skus.size_weight.to_numpy()
        fmt = self.locs.format.map(FORMAT_FACTOR).to_numpy() * self.locs.size_factor.to_numpy()
        units = BASE_UNITS_PER_STORE_DAY * p.demand_scale * fmt
        # renormalise per location so each location's carried assortment sells its target volume
        raw = carried * sku_share[None, :]
        denom = raw.sum(axis=1, keepdims=True)
        denom[denom == 0] = 1
        self.base = raw / denom * units[:, None]                     # [loc, sku]

        self.launch = pd.to_datetime(self.skus.launch_date).to_numpy().astype("datetime64[D]")
        self.category = self.skus.category.to_numpy()
        self.lifecycle = self.skus.lifecycle.to_numpy()
        self.is_hero = (self.skus.style_id == HERO_STYLE).to_numpy()
        self.hero_surge = self.is_hero & self.skus["size"].isin(HERO_SURGE_SIZES).to_numpy()
        ev = w.events[w.events.market_id == market_id]
        self.events = [(np.datetime64(e.start_date), np.datetime64(e.end_date),
                        None if e.categories == "ALL" else set(e.categories.split(",")), e.demand_uplift)
                       for e in ev.itertuples()]

    def day_factors(self, d: date, scenario: bool = True) -> np.ndarray:
        """Multiplicative factor per sku for calendar day d (season, weekday, lifecycle, events, scenario)."""
        w = self.w
        mi = d.month - 1 if self.hemisphere == "N" else (d.month + 5) % 12
        f = np.full(len(self.skus), MONTH_FACTOR_N[mi] * WEEKDAY_FACTOR[d.weekday()])
        d64 = np.datetime64(d)
        age = (d64 - self.launch).astype(int)
        life = np.where(age < 0, 0.0, np.where(age < 30, 1.0 + 0.8 * (1 - age / 30), 1.0))
        life = np.where(age > 540, 0.6, life)
        life = np.where(self.lifecycle == "MARKDOWN", life * 1.25, life)
        f = f * life
        for start, end, cats, up in self.events:
            if start <= d64 <= end:
                f = f * (np.where(np.isin(self.category, list(cats)), up, 1.0) if cats else up)
        if not scenario:
            return f
        sc = w.scenario
        hero_day = (d64 - np.datetime64(sc["hero_launch"])).astype(int)
        if self.market_id == sc["surge_market"] and hero_day >= sc["surge_from_day"]:
            f = np.where(self.hero_surge, f * sc["surge_sales"], f)
        if self.market_id == sc["slow_market"]:
            f = np.where(self.is_hero, f * sc["slow_sales"], f)
        return f

    def rates(self, days: list[date], scenario: bool = True) -> np.ndarray:
        return np.stack([self.base * self.day_factors(d, scenario)[None, :] for d in days])


def month_starts(start: date, end: date) -> list[date]:
    """First day of every calendar month overlapping [start, end]."""
    out, cur = [], date(start.year, start.month, 1)
    while cur <= end:
        out.append(cur)
        cur = date(cur.year + (cur.month == 12), cur.month % 12 + 1, 1)
    return out


def days_in_chunk(w: World, month_start: date) -> list[date]:
    p = w.params
    nxt = date(month_start.year + (month_start.month == 12), month_start.month % 12 + 1, 1)
    first = max(month_start, p.history_start)
    last = min(nxt - timedelta(days=1), p.as_of_date - timedelta(days=1))
    return [first + timedelta(days=i) for i in range((last - first).days + 1)]


_DEMAND: dict = {}


def demand_for(market_id: str) -> MarketDemand:
    if market_id not in _DEMAND:
        _DEMAND[market_id] = MarketDemand(world, market_id)
    return _DEMAND[market_id]

## Output helpers: Unity Catalog DDL, row cap, write / verify

In [ ]:
MODE = dbutils.widgets.get("mode")
catalog, schema = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")
MAX_ROWS = {t: int(dbutils.widgets.get(f"max_rows_{t}") or 0) for t in ("sell_through_daily", "digital_intent_daily")}
fq = lambda t: f"`{catalog}`.`{schema}`.`{t}`"

from pyspark.sql import functions as F
from pyspark.sql.types import DateType, DoubleType, LongType, StringType, StructField, StructType, TimestampType

TYPES = {"date": DateType(), "string": StringType(), "bigint": LongType(), "double": DoubleType(),
         "timestamp": TimestampType()}
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
if MODE == "write":
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}` "
              "COMMENT 'SYNTHETIC Kinetiq batch sources (DS-01, DS-02) for the ApparelIQ demo'")


def struct(cols):
    return StructType([StructField(c, TYPES[t], True) for c, t in cols.items()])


def create_table(name, cols, cluster, comment):
    body = ",\n  ".join(f"`{c}` {t.upper()}" for c, t in cols.items())
    spark.sql(f"""CREATE OR REPLACE TABLE {fq(name)} (
  {body}
) CLUSTER BY ({", ".join(cluster)})
COMMENT '{comment}'
TBLPROPERTIES ('appareliq.synthetic' = 'true', 'appareliq.params_fingerprint' = '{params.fingerprint()}',
               'appareliq.max_rows' = '{MAX_ROWS[name]}')""")


def cap_rows(df, table, market_id, quota):
    """Deterministic random sample of at most `quota` rows (original order kept)."""
    if quota is None or len(df) <= quota:
        return df
    seed = int(rng_for(params.seed, "cap", table, market_id).integers(1 << 31))
    return df.sample(n=max(quota, 0), random_state=seed).sort_index().reset_index(drop=True)


def checksum(sdf):
    """Order-independent content checksum: row count + sum of a 64-bit hash of every column."""
    r = sdf.agg(F.count(F.lit(1)).alias("n"),
                F.sum(F.xxhash64(*sdf.columns).cast("decimal(38,0)")).alias("h")).first()
    return int(r["n"]), int(r["h"] or 0)


def save(table, cols, chunks):
    """chunks: iterator of (market_id, pandas frame), one per market. Writes or verifies; returns the row count."""
    cap = MAX_ROWS[table] or None
    left_rows, left_markets = cap, len(world.market_ids)
    n_total, h_total, t0 = 0, 0, time.time()
    for market_id, pdf in chunks:
        quota = None if cap is None else -(-left_rows // left_markets)
        pdf = cap_rows(pdf, table, market_id, quota)
        left_markets -= 1
        if cap is not None:
            left_rows -= len(pdf)
        if len(pdf):
            pdf = pdf.copy()
            pdf["_extracted_at"] = pd.to_datetime(pdf["_extracted_at"], utc=True).dt.tz_localize(None)
            sdf = spark.createDataFrame(pdf, struct(cols))
            if MODE == "write":
                sdf.write.mode("append").saveAsTable(fq(table))
            else:
                h_total += checksum(sdf)[1]
        n_total += len(pdf)
        print(f"{table} {market_id}: {len(pdf):>10,} rows  elapsed={time.time() - t0:,.0f}s")
    if MODE == "verify":
        n_tab, h_tab = checksum(spark.table(fq(table)))
        ok = (n_tab, h_tab) == (n_total, h_total)
        print(f"VERIFY {table}: generated={n_total:,} table={n_tab:,} checksum {'MATCH' if ok else 'DIFFERENT'}")
        VERIFY[table] = "MATCH" if ok else f"DIFFERENT (generated {n_total}, table {n_tab})"
    return n_total


VERIFY: dict = {}
totals: dict = {}
SELL_THROUGH_COLUMNS = {
    "business_date": "date", "location_id": "string", "channel": "string", "style_id": "string",
    "color": "string", "size": "string", "units_sold": "bigint", "units_returned": "bigint",
    "full_price_units": "bigint", "net_sales_local": "double", "currency": "string", "net_sales_usd": "double",
    "_batch_id": "string", "_extracted_at": "timestamp", "record_label": "string",
}
INTENT_COLUMNS = {
    "business_date": "date", "style_id": "string", "color": "string", "market_id": "string",
    "search_volume": "bigint", "pdp_views": "bigint", "add_to_cart": "bigint", "notify_me": "bigint",
    "_batch_id": "string", "_extracted_at": "timestamp", "record_label": "string",
}


def chunk_keys(w: World) -> list[tuple[str, str]]:
    p = w.params
    months = month_starts(p.history_start, p.as_of_date - timedelta(days=1))
    return [(m, ms.isoformat()) for m in w.market_ids for ms in months]


def _extracted(d: date, minutes: np.ndarray) -> pd.Series:
    base = datetime.combine(d + timedelta(days=1), dtime(2, 15), tzinfo=timezone.utc)
    return pd.to_datetime([base + timedelta(minutes=int(x)) for x in minutes], utc=True)


def _empty(cols: dict) -> pd.DataFrame:
    return pd.DataFrame({c: pd.Series(dtype="object") for c in cols})


def months_of(market_id):
    return [m for mk, m in chunk_keys(world) if mk == market_id]

## Table `sell_through_daily` (DS-01)

Daily POS + e-commerce sell-through per location and style-color-size: Poisson units from the demand model, returns,
full-price vs promo units and local / USD net sales. Includes the planted data-quality issues the Silver layer must
handle: DQ-06 restatement (hero rows under-stated, corrected by a `_r1` batch), DQ-01 duplicates, DQ-02 unknown
color / location, DQ-03 invalid size (`dq_defect_rate`).

In [ ]:
ZERO_DECIMAL = {"JPY", "KRW"}
PROMO_DEPTH = {"CORE": 0.20, "LAUNCH": 0.0, "MARKDOWN": 0.30}
FACTORY_DEPTH = 0.35
RETURN_RATE = {"FOOTWEAR": 0.055, "APPAREL": 0.085}


def sell_through_chunk(w: World, market_id: str, month_start: str, md: MarketDemand | None = None) -> pd.DataFrame:
    p = w.params
    days = days_in_chunk(w, date.fromisoformat(month_start))
    if not days:
        return _empty(SELL_THROUGH_COLUMNS)
    md = md or MarketDemand(w, market_id)
    r = rng_for(p.seed, "st", market_id, month_start)
    lam = md.rates(days)
    units = r.poisson(lam)
    di, li, si = np.nonzero(units)
    u = units[di, li, si]
    sk = md.skus.iloc[si]
    loc = md.locs.iloc[li]
    m = w.markets.set_index("market_id").loc[market_id]
    price = w.prices[w.prices.market_id == market_id].set_index("style_id").full_price_local
    full = price.reindex(sk.style_id).to_numpy()
    factory = (loc.format == "FACTORY").to_numpy()
    life = sk.lifecycle.to_numpy()
    rr = sk.division.map(RETURN_RATE).to_numpy() * np.where(factory, 0.6, 1.0)
    ret = r.binomial(u, rr)
    fp_share = np.where(factory | (life == "MARKDOWN"), 0.0, np.where(life == "LAUNCH", 0.97, 0.86))
    fp = r.binomial(u, fp_share)
    depth = np.where(factory, FACTORY_DEPTH, pd.Series(life).map(PROMO_DEPTH).to_numpy())
    md_price = full * (1 - depth)
    avg_price = (fp * full + (u - fp) * md_price) / u
    net = fp * full + (u - fp) * md_price - ret * avg_price
    dec = 0 if m.currency in ZERO_DECIMAL else 2
    dates = np.array(days, dtype=object)[di]
    df = pd.DataFrame({
        "business_date": dates,
        "location_id": loc.location_id.to_numpy(),
        "channel": np.where(loc.location_type.to_numpy() == "ECOM", "ECOM", "STORE"),
        "style_id": sk.style_id.to_numpy(), "color": sk.color_code.to_numpy(), "size": sk["size"].to_numpy(),
        "units_sold": u.astype("int64"), "units_returned": ret.astype("int64"), "full_price_units": fp.astype("int64"),
        "net_sales_local": np.round(net, dec), "currency": m.currency,
        "net_sales_usd": np.round(net / m.fx_usd_to_local, 2),
    })
    df["_batch_id"] = ["st_" + d.strftime("%Y%m%d") for d in df.business_date]
    mins = r.integers(0, 45, len(days))
    ext = {d: _extracted(d, mins[i:i + 1])[0] for i, d in enumerate(days)}
    df["_extracted_at"] = df.business_date.map(ext)
    df["record_label"] = SYNTHETIC_LABEL
    df = _restatement(w, df)
    df = _defects(w, df, r)
    return df[list(SELL_THROUGH_COLUMNS)].reset_index(drop=True)


def _restatement(w: World, df: pd.DataFrame) -> pd.DataFrame:
    """DQ-06: hero rows for one day are under-stated in the first extract and corrected by batch *_r1."""
    rd = date.fromisoformat(w.scenario["restated_date"])
    mask = (df.business_date == rd) & (df.style_id == HERO_STYLE) & (df.units_sold >= 2)
    if not mask.any():
        return df
    corrected = df[mask].copy()
    corrected["_batch_id"] = corrected["_batch_id"] + "_r1"
    corrected["_extracted_at"] = pd.Timestamp(datetime.combine(rd + timedelta(days=2), dtime(9, 30)), tz="UTC")
    orig = df.loc[mask]
    scale = (orig.units_sold - 1) / orig.units_sold
    df.loc[mask, "net_sales_local"] = (orig.net_sales_local * scale).round(2)
    df.loc[mask, "net_sales_usd"] = (orig.net_sales_usd * scale).round(2)
    df.loc[mask, "units_sold"] = orig.units_sold - 1
    df.loc[mask, "full_price_units"] = np.minimum(orig.full_price_units, orig.units_sold - 1)
    df.loc[mask, "units_returned"] = np.minimum(orig.units_returned, orig.units_sold - 1)
    return pd.concat([df, corrected], ignore_index=True)


def _defects(w: World, df: pd.DataFrame, r: np.random.Generator) -> pd.DataFrame:
    """DQ-01 duplicates, DQ-02 unknown color / location, DQ-03 invalid / blank size."""
    rate = w.params.dq_defect_rate
    if rate <= 0 or df.empty:
        return df
    out = [df]
    for kind in ("dup", "color", "loc", "size"):
        n = int(r.binomial(len(df), rate))
        if n == 0:
            continue
        bad = df.iloc[r.choice(len(df), n, replace=False)].copy()
        if kind == "color":
            bad["color"] = "999"
        elif kind == "loc":
            bad["location_id"] = "ST-XX-9999"
        elif kind == "size":
            bad["size"] = np.where(r.random(n) < 0.5, "10.25", "")
        out.append(bad)
    return pd.concat(out, ignore_index=True)


def sell_through_daily_chunks():
    for market_id in world.market_ids:
        md = demand_for(market_id)
        frames = [sell_through_chunk(world, market_id, m, md) for m in months_of(market_id)]
        yield market_id, pd.concat(frames, ignore_index=True)


if MODE == "write":
    create_table("sell_through_daily", SELL_THROUGH_COLUMNS, ["business_date", "location_id"],
                 "SYNTHETIC DS-01 daily POS + e-commerce sell-through (raw extract incl. restatements and planted defects)")
totals["sell_through_daily"] = save("sell_through_daily", SELL_THROUGH_COLUMNS, sell_through_daily_chunks())

## Table `digital_intent_daily` (DS-02)

Daily digital demand intent per style-color and market: search volume, PDP views, add-to-cart and notify-me.
Scenario signals: US hero surge, GB slow sell-through, JP event spike, pre-launch buzz, BR stale feed (DQ-10, last
2 days missing) and DQ-02 unknown color `999`.

In [ ]:
def intent_chunk(w: World, market_id: str, month_start: str, md: MarketDemand | None = None) -> pd.DataFrame:
    p = w.params
    sc = w.scenario
    days = days_in_chunk(w, date.fromisoformat(month_start))
    if market_id == sc["stale_market"]:                                  # DQ-10 stale feed
        cutoff = p.as_of_date - timedelta(days=sc["stale_days"])
        days = [d for d in days if d < cutoff]
    if not days:
        return _empty(INTENT_COLUMNS)
    md = md or MarketDemand(w, market_id)
    r = rng_for(p.seed, "di", market_id, month_start)
    key = md.skus.style_id + "|" + md.skus.color_code
    codes, uniq = pd.factorize(key)
    n_sc = len(uniq)
    style = uniq.str.split("|").str[0].to_numpy()
    color = uniq.str.split("|").str[1].to_numpy()
    launch = md.skus.groupby(codes).launch_date.first().to_numpy().astype("datetime64[D]")
    is_hero = style == HERO_STYLE
    hero_launch = np.datetime64(sc["hero_launch"])
    rows = []
    for d in days:
        exp = (md.base * md.day_factors(d, scenario=False)[None, :]).sum(axis=0)
        sc_exp = np.bincount(codes, weights=exp, minlength=n_sc)
        age = (np.datetime64(d) - launch).astype(int)
        pre = (age >= -5) & (age < 0)
        if pre.any():                                                     # pre-launch buzz
            at_launch = (md.base * md.day_factors(d + timedelta(days=6), scenario=False)[None, :]).sum(axis=0)
            sc_exp = np.where(pre, 0.5 * np.bincount(codes, weights=at_launch, minlength=n_sc), sc_exp)
        mult = np.ones(n_sc)
        hday = int((np.datetime64(d) - hero_launch).astype(int))
        if market_id == sc["surge_market"] and hday >= sc["surge_from_day"]:
            mult[is_hero] = sc["surge_intent"]
        if market_id == sc["slow_market"]:
            mult[is_hero] = sc["slow_intent"]
        ev = w.events[(w.events.market_id == market_id) & (w.events.event_name == sc["spike_event"])]
        if market_id == sc["spike_market"] and ((ev.start_date <= d) & (ev.end_date >= d)).any():
            mult[is_hero] = sc["spike_intent"]
        live = (age >= -5) & (sc_exp > 0)
        search = r.poisson(sc_exp * 28 * mult * r.lognormal(0, 0.12, n_sc))
        pdp = r.binomial(search * 3, 0.82)
        atc = r.binomial(pdp, np.where(pre, 0.0, 0.075))
        notify = r.binomial(pdp, np.where(pre, 0.14, 0.006))
        rows.append(pd.DataFrame({"business_date": d, "style_id": style[live], "color": color[live],
                                  "market_id": market_id, "search_volume": search[live].astype("int64"),
                                  "pdp_views": pdp[live].astype("int64"), "add_to_cart": atc[live].astype("int64"),
                                  "notify_me": notify[live].astype("int64"),
                                  "_batch_id": "di_" + d.strftime("%Y%m%d"),
                                  "_extracted_at": _extracted(d, r.integers(30, 70, 1))[0]}))
    df = pd.concat(rows, ignore_index=True)
    df["record_label"] = SYNTHETIC_LABEL
    if w.params.dq_defect_rate > 0 and len(df):                          # DQ-02 unknown colour in intent feed
        n = max(1, int(r.binomial(len(df), w.params.dq_defect_rate)))
        bad = df.iloc[r.choice(len(df), n, replace=False)].copy()
        bad["color"] = "999"
        df = pd.concat([df, bad], ignore_index=True)
    return df[list(INTENT_COLUMNS)]


def digital_intent_daily_chunks():
    for market_id in world.market_ids:
        md = demand_for(market_id)
        frames = [intent_chunk(world, market_id, m, md) for m in months_of(market_id)]
        yield market_id, pd.concat(frames, ignore_index=True)


if MODE == "write":
    create_table("digital_intent_daily", INTENT_COLUMNS, ["business_date", "market_id"],
                 "SYNTHETIC DS-02 daily digital demand intent (search, PDP views, add-to-cart, notify-me)")
totals["digital_intent_daily"] = save("digital_intent_daily", INTENT_COLUMNS, digital_intent_daily_chunks())
print(totals)

## Validation

In [ ]:
counts = {t: spark.table(fq(t)).count() for t in totals}
if MODE == "write":
    assert counts == totals, (counts, totals)
else:
    assert all(v == "MATCH" for v in VERIFY.values()), VERIFY
assert spark.table(fq("sell_through_daily")).where("record_label <> 'SYNTHETIC'").count() == 0
display(spark.sql(f"""
SELECT l.market, count(*) AS rows, sum(units_sold) AS units, round(sum(net_sales_usd)) AS net_sales_usd,
       min(business_date) AS first_day, max(business_date) AS last_day
FROM (SELECT substr(location_id, 4, 2) AS market, * FROM {fq('sell_through_daily')}) l
GROUP BY l.market ORDER BY units DESC"""))

In [ ]:
# Hero launch (HM9443 Men's AeroGlide 5): US core sizes vs. the rest, last 14 days
display(spark.sql(f"""
SELECT substr(location_id, 4, 2) AS market,
       CASE WHEN size IN ('9','9.5','10','10.5','11') THEN 'core 9-11' ELSE 'other' END AS size_band,
       sum(units_sold) AS units
FROM {fq('sell_through_daily')}
WHERE style_id = 'HM9443' AND business_date >= date_sub(DATE'{params.as_of}', 14)
GROUP BY ALL ORDER BY market, size_band"""))

In [ ]:
dbutils.notebook.exit(json.dumps({"mode": MODE, "fingerprint": params.fingerprint(), **counts,
                                  **({"verify": VERIFY} if VERIFY else {})}))